# Agentic RAG, the LangGraph way — Zero to Hero

*A branch of the **Agentic RAG Research Assistant**, rebuilt with LangGraph.*

This single notebook does two things at once:

1. **Teaches LangGraph from zero** — state, nodes, edges, conditional edges,
   tools, and the ReAct agent loop — following the arc of the *"LangGraph Complete
   Course For Beginners (From ZERO To HERO)"* crash course.
2. **Rebuilds our exact project** — question-answering over the same 2,012 arXiv
   papers — but this time the agent is orchestrated as an explicit **graph**
   instead of the hand-written `while` loop in [`../src/agent.py`](../src/agent.py).

**Why do this at all?** The parent project deliberately wrote its agent loop *from
scratch* to keep it fully inspectable. LangGraph is what you reach for when that
loop grows up: it turns the control flow into a **directed graph** you can draw,
checkpoint, stream, and extend — without hiding the logic. Same retriever, same
corpus, same finding; a different, production-shaped way to wire the agent.

> **The one big idea.** In LangGraph you stop writing `if`/`while` control flow by
> hand and instead *declare a graph*: **nodes** are steps that read and update a
> shared **state**, **edges** say what runs next, and **conditional edges** are the
> branches. The runtime walks the graph for you. Our from-scratch loop and this
> graph do the same thing — LangGraph just makes the shape explicit.

Nothing here re-implements retrieval: it imports the **same** `VectorStore` the
rest of the project uses, so this is genuinely the same project with a new spine.

## 0 · Setup

We reuse the parent project's `src/` package (config, the FAISS vector store, the
embedder, prompt helpers). The vector index must already be built — if you haven't,
run `python ../run_pipeline.py` once from this folder's parent.

In [1]:
import sys, json
from pathlib import Path
from typing import TypedDict, Annotated
import operator

# The project root is this notebook's grandparent (langgraph-branch/ -> project/).
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config
from src.vectorstore import VectorStore

store = VectorStore.load()          # the SAME index the from-scratch agent uses
print("Project root:", ROOT)
print("Vector index loaded:", store.size, "chunks")
print("API key present:", config.api_key_available(), "(LLM nodes fall back gracefully without it)")

Project root: D:\Claude\Data Science Projects\01-projects\agentic-rag-research-assistant
Vector index loaded: 4299 chunks
API key present: False (LLM nodes fall back gracefully without it)


## 1 · What is LangGraph, and why a graph?

An **agent** is an LLM that takes actions in a loop: think → act → observe →
repeat. You can write that loop by hand (we did, in `src/agent.py`). LangGraph
offers a different mental model: describe the agent as a **state machine / directed
graph**.

Three primitives — that's the whole course in one table:

| Primitive | What it is | In plain Python it's like… |
|---|---|---|
| **State** | A shared dictionary every step reads and writes | the variables in your loop |
| **Node** | A function `state -> partial state update` | one step of the loop body |
| **Edge** | "after node A, go to node B" | the order of your statements |
| **Conditional edge** | "after A, look at the state and pick B *or* C" | an `if`/`while` branch |

You **add nodes and edges to a `StateGraph`, then `.compile()`** it into a runnable.
Calling `.invoke(initial_state)` walks the graph from `START` to `END`, threading
the state through each node. Let's build the smallest possible one.

## 2 · State — the shared memory

State is a `TypedDict`. Each key is a channel of data flowing through the graph.
A node returns a **partial** dict; LangGraph merges it into the running state.

One subtlety worth learning early: by default a returned key **overwrites** the
old value. If instead you want a node to **append** (e.g. accumulate retrieved
documents across several retrieval steps), you annotate that key with a **reducer**
— `Annotated[list, operator.add]` means "merge by adding lists". We'll use both.

In [2]:
class HelloState(TypedDict):
    # a plain channel: last writer wins
    name: str
    # a reduced channel: every node's list gets concatenated, not overwritten
    steps: Annotated[list[str], operator.add]

print("HelloState channels:", HelloState.__annotations__)

HelloState channels: {'name': <class 'str'>, 'steps': typing.Annotated[list[str], <built-in function add>]}


## 3 · Nodes + edges — your first graph

A node is just a function. Here are two trivial ones. Notice each **returns a dict**
that updates the state — a node never mutates state in place, it *describes* the
update. Then we wire `START -> greet -> shout -> END` and compile.

This graph has **no LLM**, so it runs anywhere, no key needed — perfect for seeing
the mechanics.

In [3]:
from langgraph.graph import StateGraph, START, END

def greet(state: HelloState) -> dict:
    return {"steps": [f"greeted {state['name']}"]}     # appended (reducer)

def shout(state: HelloState) -> dict:
    return {"name": state["name"].upper(), "steps": ["shouted"]}   # name overwritten

g = StateGraph(HelloState)
g.add_node("greet", greet)
g.add_node("shout", shout)
g.add_edge(START, "greet")     # every graph needs an edge out of START
g.add_edge("greet", "shout")
g.add_edge("shout", END)       # ...and into END
hello = g.compile()

result = hello.invoke({"name": "arxiv", "steps": []})
print(result)

{'name': 'ARXIV', 'steps': ['greeted arxiv', 'shouted']}


See it: `name` was **overwritten** to `ARXIV` (last writer wins), while `steps`
**accumulated** both entries (the `operator.add` reducer). That overwrite-vs-append
distinction is exactly what we'll use to accumulate retrieved documents later.

### Drawing the graph

Anything you build in LangGraph, you can *see*. `get_graph().draw_mermaid()` emits
a Mermaid diagram of the topology — one of the real payoffs of declaring control
flow instead of writing it inline.

In [4]:
print(hello.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	greet(greet)
	shout(shout)
	__end__([<p>__end__</p>]):::last
	__start__ --> greet;
	greet --> shout;
	shout --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4 · Conditional edges — branching and loops

This is where a graph earns its keep. A **conditional edge** runs a small router
function that reads the state and returns the *name* of the next node. That's how
you express `if`/`else` **and** loops (an edge that can point back to an earlier
node).

Toy example: count up to a target, looping until we get there. The router returns
either `"increment"` (loop again) or `END` (stop) — a loop drawn as a graph.

In [5]:
class CountState(TypedDict):
    n: int
    target: int

def increment(state: CountState) -> dict:
    return {"n": state["n"] + 1}

def keep_going(state: CountState) -> str:
    # a router returns the NAME of the next node (or END)
    return "increment" if state["n"] < state["target"] else END

cg = StateGraph(CountState)
cg.add_node("increment", increment)
cg.add_edge(START, "increment")
cg.add_conditional_edges("increment", keep_going, {"increment": "increment", END: END})
counter = cg.compile()

print("counted to:", counter.invoke({"n": 0, "target": 5})["n"])
print()
print(counter.get_graph().draw_mermaid())

counted to: 5

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	increment(increment)
	__end__([<p>__end__</p>]):::last
	__start__ --> increment;
	increment -.-> __end__;
	increment -.-> increment;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



That self-loop (`increment -> increment`) is the graph equivalent of our agent's
`for _ in range(MAX_AGENT_STEPS)` loop. Hold that thought — the RAG agent below is
exactly this shape, with "retrieve" in place of "increment" and "do I have enough
evidence?" as the router.

## 5 · Tools and the LLM node

An agent needs (a) an LLM and (b) tools it can call. In LangGraph the LLM is a
LangChain chat model; here `ChatAnthropic` wrapping the same Claude model our
project already uses (`config.GEN_MODEL`). A **tool** is a plain function decorated
with `@tool` — LangChain turns its signature + docstring into the schema the model
sees, just like the hand-written `SEARCH_TOOL` schema in `src/agent.py`.

Our one tool is the same retriever the whole project shares.

In [6]:
from langchain_core.tools import tool

@tool
def search_corpus(query: str) -> str:
    """Search the arXiv paper corpus for passages relevant to a query.
    Returns the most similar passages with their arXiv ids. Call multiple times
    with different focused queries to gather evidence for a multi-part question."""
    hits = store.search(query, k=config.TOP_K)
    if not hits:
        return "No passages found."
    return "\n\n".join(f"[{h.arxiv_id}] {h.title}\n{h.text}" for h in hits)

# Quick local smoke test (retrieval is local — no key needed):
print(search_corpus.invoke({"query": "dense passage retrieval"})[:300], "...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[2004.04906] Dense Passage Retrieval for Open-Domain Question Answering
Dense Passage Retrieval for Open-Domain Question Answering

Open-domain question answering relies on efficient passage retrieval to select candidate contexts, where traditional sparse vector space models, such as TF-IDF or BM25, ...


In [7]:
# The LLM. Constructing it does NOT call the API (that happens on invoke),
# so this cell is safe even without a key.
llm_model = None
if config.api_key_available():
    from langchain_anthropic import ChatAnthropic
    llm_model = ChatAnthropic(model=config.GEN_MODEL, max_tokens=config.MAX_ANSWER_TOKENS)
    print("ChatAnthropic ready:", config.GEN_MODEL)
else:
    print("No API key — the LLM nodes below use a deterministic fallback so the")
    print("graph still runs end to end. Set ANTHROPIC_API_KEY for real generation.")

No API key — the LLM nodes below use a deterministic fallback so the
graph still runs end to end. Set ANTHROPIC_API_KEY for real generation.


## 6 · The project — Agentic RAG as an explicit graph

Now we rebuild our agent. Recall the finding from the parent project: **single-shot
retrieval nails single-hop questions (recall@5 = 1.00) but drops to 0.60 on
multi-hop ones** because one search can't cover two pieces of evidence. The agent's
job is to fix that by retrieving *iteratively*.

Here is that agent as a graph:

```mermaid
graph LR
  START --> plan
  plan --> retrieve
  retrieve --> decide
  decide -- need more evidence --> plan
  decide -- enough --> generate
  generate --> END
```

The state and the four nodes:

- **state** carries the question, the running list of documents (a *reduced*
  channel — it accumulates across loops), an iteration counter, and the final
  answer.
- **plan** — ask the LLM to propose the next focused search query (decomposition).
  Fallback without a key: just use the question itself.
- **retrieve** — run `search_corpus` for the planned query; append the hits.
- **decide** — the conditional edge: have we gathered evidence from enough distinct
  papers, or hit the step cap? If not, loop back to **plan**; else go to **generate**.
- **generate** — write the grounded, cited answer from everything retrieved.

This is the same behaviour as `src/agent.py`, re-expressed as a graph you can draw.

In [8]:
from src import rag   # reuse the grounded prompt + citation helpers

class RAGState(TypedDict):
    question: str
    query: str                                    # the current search query
    documents: Annotated[list, operator.add]      # accumulates across loops
    iterations: int
    generation: str

MAX_LOOPS = config.MAX_AGENT_STEPS

def plan(state: RAGState) -> dict:
    """Choose the next focused query. LLM-driven; falls back to the question."""
    q = state["question"]
    already = state.get("iterations", 0)
    if llm_model is not None:
        seen = {d.arxiv_id for d in state.get("documents", [])}
        prompt = (
            f"You are planning searches to answer: {q}\n"
            f"You have already retrieved papers: {sorted(seen) or 'none yet'}.\n"
            "Propose ONE focused search query for a piece of evidence you still "
            "need. Reply with only the query text."
        )
        query = llm_model.invoke(prompt).content.strip()
    else:
        # Deterministic fallback: first loop uses the whole question, later loops
        # nudge toward a different phrasing so the demo still shows iteration.
        query = q if already == 0 else f"{q} method details"
    return {"query": query, "iterations": already + 1}

def retrieve(state: RAGState) -> dict:
    """Run the shared retriever for the planned query; append the hits."""
    hits = store.search(state["query"], k=config.TOP_K)
    return {"documents": hits}

def decide(state: RAGState) -> str:
    """Conditional edge: loop back to plan, or move on to generate."""
    n_papers = len({d.arxiv_id for d in state["documents"]})
    if state["iterations"] >= MAX_LOOPS:
        return "generate"                 # step cap — stop looping
    if n_papers >= 2:
        return "generate"                 # enough distinct evidence gathered
    return "plan"                          # need more — search again

def generate(state: RAGState) -> dict:
    """Write the grounded, cited answer from all retrieved documents."""
    context = rag.format_context(state["documents"])
    if llm_model is not None:
        user = (f"Context passages:\n\n{context}\n\nQuestion: {state['question']}\n\n"
                "Answer using only the passages above, and cite the arXiv ids you used.")
        msg = [("system", rag.SYSTEM_PROMPT), ("human", user)]
        answer = llm_model.invoke(msg).content
    else:
        # Fallback: an honest extractive stub so the graph completes without a key.
        cited = sorted({d.arxiv_id for d in state["documents"]})
        answer = ("[fallback answer - no API key] Retrieved evidence from papers "
                  + ", ".join(f"[{c}]" for c in cited) + ". Set ANTHROPIC_API_KEY "
                  "for a real grounded generation.")
    return {"generation": answer}

print("nodes defined:", [f.__name__ for f in (plan, retrieve, decide, generate)])

nodes defined: ['plan', 'retrieve', 'decide', 'generate']


### Wire and compile the graph

In [9]:
rag_graph = StateGraph(RAGState)
rag_graph.add_node("plan", plan)
rag_graph.add_node("retrieve", retrieve)
rag_graph.add_node("generate", generate)

rag_graph.add_edge(START, "plan")
rag_graph.add_edge("plan", "retrieve")
rag_graph.add_conditional_edges("retrieve", decide, {"plan": "plan", "generate": "generate"})
rag_graph.add_edge("generate", END)

agent_app = rag_graph.compile()
print(agent_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	retrieve(retrieve)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	plan --> retrieve;
	retrieve -.-> generate;
	retrieve -.-> plan;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Run it — and watch it iterate

We stream the state as it flows through the graph, so you can *see* each search the
agent decides to run — the LangGraph equivalent of the search trace our from-scratch
agent prints. This runs with or without a key (fallback generation kicks in).

In [10]:
question = ("How does retrieval-augmented generation combine a dense passage "
            "retriever with a sequence-to-sequence generator?")

init = {"question": question, "query": "", "documents": [], "iterations": 0, "generation": ""}

print("=== streaming graph execution ===")
for event in agent_app.stream(init):
    for node_name, update in event.items():
        if node_name == "plan":
            print(f"[plan]     query -> {update['query']!r}")
        elif node_name == "retrieve":
            ids = sorted({d.arxiv_id for d in update["documents"]})
            print(f"[retrieve] hits -> {ids}")
        elif node_name == "generate":
            print(f"[generate] answer ->\n{update['generation'][:400]}")

=== streaming graph execution ===
[plan]     query -> 'How does retrieval-augmented generation combine a dense passage retriever with a sequence-to-sequence generator?'
[retrieve] hits -> ['2004.04906', '2005.11401', '2607.17598', '2607.18756', '2607.21324']
[generate] answer ->
[fallback answer - no API key] Retrieved evidence from papers [2004.04906], [2005.11401], [2607.17598], [2607.18756], [2607.21324]. Set ANTHROPIC_API_KEY for a real grounded generation.


## 7 · The fast path — `create_react_agent`

Everything above we wired by hand to *learn* it. In practice LangGraph ships a
prebuilt **ReAct agent** that builds the tool-calling loop for you from a model and
a list of tools — the "hero" payoff of the course. It's the same reason-act-observe
loop, one line to construct.

In [11]:
react_app = None
if llm_model is not None:
    from langgraph.prebuilt import create_react_agent
    react_app = create_react_agent(llm_model, tools=[search_corpus])
    print(react_app.get_graph().draw_mermaid())
    out = react_app.invoke({"messages": [("user", question)]})
    print("\n=== ReAct agent answer ===")
    print(out["messages"][-1].content[:600])
else:
    print("create_react_agent needs a live model — set ANTHROPIC_API_KEY to run it.")
    print("Its graph is the canonical agent loop: agent -> (tools -> agent)* -> END,")
    print("i.e. the same shape as our hand-built graph in section 6.")

create_react_agent needs a live model — set ANTHROPIC_API_KEY to run it.
Its graph is the canonical agent loop: agent -> (tools -> agent)* -> END,
i.e. the same shape as our hand-built graph in section 6.


## 8 · How this maps back to the parent project

Same project, three ways to wire the *same* agent over the *same* retriever:

| Concern | `src/agent.py` (from scratch) | This notebook (LangGraph) |
|---|---|---|
| Control flow | a hand-written `for` loop | a compiled `StateGraph` you can draw |
| "Search again?" | an `if` on tool-use blocks | a **conditional edge** (`decide`) |
| Evidence pool | a `dict` de-duped by chunk id | a **reduced** state channel (`operator.add`) |
| Tool schema | hand-written JSON in `SEARCH_TOOL` | `@tool` infers it from the function |
| Step cap | `MAX_AGENT_STEPS` | same constant, checked in `decide` |
| Reasoning trace | printed from `AgentStep`s | `app.stream(...)` events |
| The retriever | `VectorStore.search` | **the exact same** `VectorStore.search` |

Neither is "better" universally. The from-scratch loop is the most transparent way
to *understand* an agent; the graph is what you want when the loop needs
checkpointing, human-in-the-loop pauses, parallel branches, or streaming to a UI —
all things LangGraph gives you once the control flow is a graph.

**The finding is unchanged**, because retrieval is unchanged: single-shot RAG
misses multi-hop evidence, and the iterative agent — whether a `while` loop or this
graph — is what recovers it. See [`../README.md`](../README.md) and the parent
capstone for the measured numbers.

## 9 · Sanity check — the graph uses the same evaluated retriever

To make "same project" concrete: the retrieval metrics that produced the finding
come from `src/evaluation.py` against this very `store`. Re-run the local half here
(no key needed) and confirm the multi-hop gap this LangGraph agent exists to close.

In [12]:
from src import evaluation as ev
import pandas as pd

rows = ev.evaluate_retrieval(store, ev.load_eval_set())
summary = pd.DataFrame(rows).groupby("hop")[["hit@k", "recall@k", "mrr"]].mean().round(3)
print(summary)
print("\nThe multi-hop recall gap below 1.0 is exactly what the iterative graph targets.")

        hit@k  recall@k    mrr
hop                           
multi     0.8       0.6  0.800
single    1.0       1.0  0.681

The multi-hop recall gap below 1.0 is exactly what the iterative graph targets.


## 10 · Recap — zero to hero

You built, from nothing:

1. **State** (`TypedDict`, plus a reducer for accumulation).
2. **Nodes** (functions returning partial state) and **edges** (`START`/`END`).
3. **Conditional edges** — branching and loops as a graph.
4. **Tools** (`@tool`) and an **LLM node** (`ChatAnthropic`).
5. **The project**: an agentic-RAG `StateGraph` that plans → retrieves → decides →
   generates, looping until it has multi-hop evidence — the same behaviour as
   `src/agent.py`, drawn as a graph.
6. The **prebuilt ReAct agent**, the one-liner for the same loop.

Same corpus, same retriever, same finding — a new, graph-shaped spine.

### Reproduce
```bash
cd ..                      # project root
python run_pipeline.py     # build the index once (local, no key)
pip install langgraph langchain-anthropic langchain-core
# then run this notebook top-to-bottom; set ANTHROPIC_API_KEY for real generation.
```